In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ravindrasinghrana/job-description-dataset")

print("Path to dataset files:", path)

100%|██████████| 457M/457M [00:07<00:00, 66.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ravindrasinghrana/job-description-dataset/versions/1


In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

In [2]:
# data = pd.read_csv(path + "/job_descriptions.csv")
data = joblib.load('jobs_data.pkl')

In [3]:
data.head()

,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile,combined_text
0,1089843540111562,5 - 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,Digital Marketing Specialist,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie...","M.Tech Social media platforms (e.g., Facebook,..."
1,398454096642776,2 - 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,Web Developer,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com...","BCA HTML, CSS, JavaScript Frontend frameworks ..."
2,481640072963533,0 - 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,Operations Manager,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P...",PhD Quality control processes and methodologie...
3,688192671473044,4 - 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,...,Network Engineer,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O...",PhD Wireless network design and architecture W...
4,117057806156508,1 - 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,...,Event Manager,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ...",MBA Event planning Conference logistics Budget...


In [4]:
data.shape

(1610462, 24)

In [5]:
data["skills"]

0          Social media platforms (e.g., Facebook, Twitte...
1          HTML, CSS, JavaScript Frontend frameworks (e.g...
2          Quality control processes and methodologies St...
3          Wireless network design and architecture Wi-Fi...
4          Event planning Conference logistics Budget man...
                                 ...                        
1615935    Mechanical engineering CAD software (e.g., Sol...
1615936    Strategic IT planning Leadership and managemen...
1615937    Mechanical engineering CAD software (e.g., Sol...
1615938    Training program coordination Training materia...
1615939    Wedding planning Venue selection Catering and ...
Name: skills, Length: 1610462, dtype: object

In [6]:
data["Qualifications"].value_counts()

Qualifications
BBA       161574
BA        161566
BCA       161215
M.Tech    161186
PhD       161109
MBA       160967
B.Tech    160886
M.Com     160833
B.Com     160774
MCA       160352
Name: count, dtype: int64

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1610462 entries, 0 to 1615939
Data columns (total 24 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   Job Id            1610462 non-null  int64  
 1   Experience        1610462 non-null  object 
 2   Qualifications    1610462 non-null  object 
 3   Salary Range      1610462 non-null  object 
 4   location          1610462 non-null  object 
 5   Country           1610462 non-null  object 
 6   latitude          1610462 non-null  float64
 7   longitude         1610462 non-null  float64
 8   Work Type         1610462 non-null  object 
 9   Company Size      1610462 non-null  int64  
 10  Job Posting Date  1610462 non-null  object 
 11  Preference        1610462 non-null  object 
 12  Contact Person    1610462 non-null  object 
 13  Contact           1610462 non-null  object 
 14  Job Title         1610462 non-null  object 
 15  Role              1610462 non-null  object 
 16  Job P

In [8]:
data.dropna(axis=0, inplace=True)

In [9]:
data.shape

(1610462, 24)

In [10]:
data["Experience"] = data["Experience"].str.replace("to", "-")

In [11]:
data["Experience"]

0          5 - 15 Years
1          2 - 12 Years
2          0 - 12 Years
3          4 - 11 Years
4          1 - 12 Years
               ...     
1615935    0 - 12 Years
1615936    2 - 14 Years
1615937    4 - 15 Years
1615938    5 - 15 Years
1615939    1 - 11 Years
Name: Experience, Length: 1610462, dtype: object

In [12]:
model = TfidfVectorizer(stop_words="english")

In [13]:
data['combined_text'] = data[["Qualifications","skills", "Experience"]].astype(str).agg(' '.join, axis=1)

In [ ]:
tfidf_matrix = model.fit_transform(data["combined_text"])

In [ ]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 25408532 stored elements and shape (1610462, 1035)>

In [ ]:
# features_names = model.get_feature_names_out()

In [ ]:
# dense_matrix = tfidf_matrix.toarray()

In [ ]:
def recommendations(user_input):
    user_vec = model.transform([user_input])
    cosine_sim = cosine_similarity(user_vec, tfidf_matrix)
    sim_jobs = cosine_sim[0].argsort()[::-1]  # Fix: use [0] to flatten to 1D
    return data.iloc[sim_jobs]


In [ ]:
recommended = recommendations("B.Tech HTML CSS JavaScript DevOps 0 - 5 years")

In [ ]:
recommended.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1610462 entries, 1115744 to 6432
Data columns (total 24 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   Job Id            1610462 non-null  int64  
 1   Experience        1610462 non-null  object 
 2   Qualifications    1610462 non-null  object 
 3   Salary Range      1610462 non-null  object 
 4   location          1610462 non-null  object 
 5   Country           1610462 non-null  object 
 6   latitude          1610462 non-null  float64
 7   longitude         1610462 non-null  float64
 8   Work Type         1610462 non-null  object 
 9   Company Size      1610462 non-null  int64  
 10  Job Posting Date  1610462 non-null  object 
 11  Preference        1610462 non-null  object 
 12  Contact Person    1610462 non-null  object 
 13  Contact           1610462 non-null  object 
 14  Job Title         1610462 non-null  object 
 15  Role              1610462 non-null  object 
 16  Jo

In [10]:
import joblib

# # Save the model
# joblib.dump(model, 'job_recommendation_model.pkl')

# # Save the TF-IDF matrix
# joblib.dump(tfidf_matrix, 'tfidf_matrix.pkl')

# Optionally, save the data DataFrame too
joblib.dump(data, 'jobs_data.pkl')


['jobs_data.pkl']

In [ ]:
# Load the model
model = joblib.load('job_recommendation_model.pkl')

# Load the TF-IDF matrix
tfidf_matrix = joblib.load('tfidf_matrix.pkl')

# Load the jobs data
data = joblib.load('jobs_data.pkl')


In [11]:
from google.colab import files
files.download('jobs_data.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>